# LAB 1 — Dataset de Avaliação com Ground-Truth
## Aula 5 — Avaliação de Pipelines RAG com RAGAS
**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**

---

## Stack alinhada ao curso (Aula 3+)

- **LLM gerador**: Groq Cloud `llama-3.1-8b-instant` (via `langchain_groq.ChatGroq`) — fallback automático Ollama `llama3.2:3b`
- **Embeddings**: Ollama `bge-m3` (dim=1024) via `langchain_ollama.OllamaEmbeddings`
- **Vector Store**: OpenSearch 3.x — índice de acórdãos TCU 2026 (Aula 4)
- **Avaliação**: dataset RAGAS-compatible (50 pares `question / answer / contexts / ground_truth`)

## Objetivo

Construir o **dataset de avaliação** que será usado nos LABs 2–6: executar o **Naive RAG** sobre 50 perguntas jurídicas e capturar `(question, answer, contexts, ground_truth)` no formato exigido pelo RAGAS.

## 1. Instalação de Dependências

In [ ]:
import subprocess, sys
for pkg in [
    'opensearch-py>=2.7', 'python-dotenv>=1.0', 'pandas', 'tqdm',
    'langchain>=0.2', 'langchain-core>=0.2',
    'langchain-groq>=0.1',     # ChatGroq
    'langchain-ollama>=0.1',   # ChatOllama + OllamaEmbeddings
    'ragas>=0.1.16', 'datasets',
]:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✓ dependências instaladas')

## 2. Carregar `.env` + Stack (Groq + Ollama + OpenSearch)

In [ ]:
import os, json, time, warnings
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

for env_path in [Path.cwd()/'.env', Path.home()/'mba-rag'/'.env',
                  Path.cwd().parent.parent/'.env']:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f'.env carregado: {env_path}')
        break
else:
    load_dotenv(override=False)

GROQ_API_KEY       = os.getenv('GROQ_API_KEY', '')
GROQ_LLM_MODEL     = os.getenv('GROQ_LLM_MODEL',   'llama-3.1-8b-instant')
OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL',    'http://localhost:11434')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL',   'llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')
OPENSEARCH_HOST    = os.getenv('OPENSEARCH_HOST',     'localhost')
OPENSEARCH_PORT    = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER    = os.getenv('OPENSEARCH_USER',     'admin')
OPENSEARCH_PASS    = os.getenv('OPENSEARCH_PASSWORD', 'admin')
EMBED_DIM          = 1024

# Índice TCU 2026 (compartilhado com Aula 4)
INDEX_NAME       = os.getenv('INDEX_NAME', 'corpus_juridico_aula4')
FALLBACK_INDEX   = 'aula5_corpus_juridico_baseline'
DATASET_PATH     = Path('..') / 'datasets' / 'corpus_avaliacao_aula5.json'
CORPUS_AULA4     = Path('..') / '..' / 'aula4' / 'datasets' / 'corpus_juridico_aula4_v2.json'
K_DOCS           = 5

from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from opensearchpy import OpenSearch

# ── LLM gerador (Groq primário, Ollama fallback) ──────────────────────────
llm = None; LLM_PROVIDER = None; LLM_MODEL = None
if GROQ_API_KEY:
    try:
        c = ChatGroq(model=GROQ_LLM_MODEL, api_key=GROQ_API_KEY,
                     temperature=0.1, max_tokens=512, timeout=30)
        c.invoke([HumanMessage(content='ok')])
        llm, LLM_PROVIDER, LLM_MODEL = c, 'groq', GROQ_LLM_MODEL
    except Exception as e:
        print(f'Groq indisponível ({e.__class__.__name__}). Fallback Ollama.')
if llm is None:
    llm = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL,
                     temperature=0.1, num_predict=512)
    LLM_PROVIDER, LLM_MODEL = 'ollama', OLLAMA_LLM_MODEL

# ── Embeddings BGE-M3 via Ollama ──────────────────────────────────────────
embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
_ = embeddings.embed_query('teste'); assert len(_) == EMBED_DIM

# ── OpenSearch ────────────────────────────────────────────────────────────
os_client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, timeout=60,
)
OS_VER = os_client.info()['version']['number']

print(f"Stack ✓ | LLM={LLM_PROVIDER}/{LLM_MODEL} | embed={OLLAMA_EMBED_MODEL}/{EMBED_DIM}d | OpenSearch {OS_VER}")

## 3. Garantir Índice de Acórdãos TCU (vector store)

Usa o índice da Aula 4 se existir; caso contrário cria e ingere o corpus `aula4/datasets/corpus_juridico_aula4_v2.json`.

In [ ]:
from tqdm.auto import tqdm

MAPPING_HIBRIDO = {
    'settings': {'index': {'knn': True, 'number_of_shards': 1, 'number_of_replicas': 0},
                  'analysis': {'analyzer': {'pt': {'type': 'standard', 'stopwords': '_portuguese_'}}}},
    'mappings': {'properties': {
        'id': {'type': 'keyword'}, 'tipo': {'type': 'keyword'},
        'titulo':   {'type': 'text', 'analyzer': 'pt'},
        'conteudo': {'type': 'text', 'analyzer': 'pt'},
        'metadata': {'type': 'object'},
        'embedding': {'type': 'knn_vector', 'dimension': EMBED_DIM,
                       'method': {'name': 'hnsw', 'space_type': 'cosinesimil', 'engine': 'faiss',
                                  'parameters': {'ef_construction': 128, 'm': 16}}}
    }}
}

if os_client.indices.exists(index=INDEX_NAME):
    cnt = os_client.count(index=INDEX_NAME)['count']
    print(f'✓ índice "{INDEX_NAME}" já existe com {cnt} documentos — usando.')
elif CORPUS_AULA4.exists():
    print(f'Índice "{INDEX_NAME}" inexistente. Criando "{FALLBACK_INDEX}" a partir do corpus Aula 4.')
    INDEX_NAME = FALLBACK_INDEX
    if os_client.indices.exists(index=INDEX_NAME):
        os_client.indices.delete(index=INDEX_NAME)
    os_client.indices.create(index=INDEX_NAME, body=MAPPING_HIBRIDO)
    with open(CORPUS_AULA4, encoding='utf-8') as f:
        corpus_docs = json.load(f)
    LIM = int(os.getenv('AULA5_LAB1_MAX_DOCS', 200))
    corpus_docs = corpus_docs[:LIM]
    BATCH = 16
    for i in tqdm(range(0, len(corpus_docs), BATCH), desc='Ingerindo TCU 2026'):
        lote = corpus_docs[i:i+BATCH]
        textos = [(d.get('titulo','') + '\n\n' + d.get('conteudo','')).strip() for d in lote]
        vecs = embeddings.embed_documents(textos)
        body = []
        for d, v in zip(lote, vecs):
            body.append({'index': {'_index': INDEX_NAME, '_id': d['id']}})
            body.append({'id': d['id'], 'tipo': d.get('tipo',''),
                         'titulo': d.get('titulo',''), 'conteudo': d.get('conteudo',''),
                         'metadata': d.get('metadata', {}), 'embedding': v})
        os_client.bulk(body=body, request_timeout=180)
    os_client.indices.refresh(index=INDEX_NAME)
    print(f'✓ {os_client.count(index=INDEX_NAME)["count"]} docs indexados em "{INDEX_NAME}"')
else:
    print(f'⚠ Sem corpus base. Pipeline retornará contexto vazio.')

## 4. Carregar Dataset de Avaliação (50 pares QA)

In [ ]:
import pandas as pd

if DATASET_PATH.exists():
    with open(DATASET_PATH, encoding='utf-8') as f:
        corpus_eval = json.load(f)
    print(f'✓ {len(corpus_eval)} pares QA carregados de {DATASET_PATH.name}')
else:
    raise FileNotFoundError(f'{DATASET_PATH} não encontrado')

df_pre = pd.DataFrame(corpus_eval)
print('\nDistribuição por tipo jurídico:')
print(df_pre['tipo'].value_counts().to_string())
print('\nDistribuição por dificuldade:')
print(df_pre['dificuldade'].value_counts().to_string())

## 5. Pipeline Naive RAG (LCEL — embed + kNN + Groq/Ollama)

In [ ]:
PROMPT_RAG = ChatPromptTemplate.from_messages([
    ('system',
     'Você é um assistente jurídico brasileiro. Responda APENAS com base nos trechos fornecidos, '
     'citando dispositivos legais quando relevante.'),
    ('human',
     'CONTEXTOS:\n{contextos}\n\nPERGUNTA:\n{pergunta}\n\nRESPOSTA:')
])
RAG_CHAIN = PROMPT_RAG | llm | StrOutputParser()

def buscar_chunks(query: str, k: int = K_DOCS) -> list[str]:
    """Retorna os top-k trechos via busca kNN no OpenSearch (embedding via Ollama BGE-M3)."""
    if not os_client.indices.exists(index=INDEX_NAME):
        return []
    vec = embeddings.embed_query(query)
    body = {'size': k,
            'query': {'knn': {'embedding': {'vector': vec, 'k': k}}},
            '_source': ['titulo', 'conteudo']}
    try:
        r = os_client.search(index=INDEX_NAME, body=body)
        return [(h['_source'].get('titulo', '') + '\n' + h['_source'].get('conteudo', '')).strip()
                 for h in r['hits']['hits']]
    except Exception as e:
        print(f'erro busca: {e}')
        return []

def naive_rag(question: str, ground_truth: str, k: int = K_DOCS) -> dict:
    ctxs = buscar_chunks(question, k=k)
    ctxs_safe = ctxs if ctxs else ['Contexto indisponível.']
    bloco = '\n\n'.join(f'[Trecho {i+1}]\n{c[:800]}' for i, c in enumerate(ctxs_safe))
    try:
        ans = RAG_CHAIN.invoke({'contextos': bloco, 'pergunta': question}).strip()
    except Exception as e:
        ans = f'[Erro geração: {e}]'
    return {'question': question, 'answer': ans,
            'contexts': ctxs_safe, 'ground_truth': ground_truth}

# Smoke test
_t = naive_rag(corpus_eval[0]['question'], corpus_eval[0]['ground_truth'])
print(f"smoke test ✓ | {len(_t['contexts'])} ctxs | answer head: {_t['answer'][:120]}...")

## 6. Gerar Respostas para os 50 Pares

In [ ]:
dataset_completo: list[dict] = []
t0 = time.time()

for par in tqdm(corpus_eval, desc='Naive RAG (50 pares)'):
    try:
        r = naive_rag(par['question'], par['ground_truth'], k=K_DOCS)
        r.update({'id': par['id'], 'tipo': par['tipo'], 'dificuldade': par['dificuldade']})
        dataset_completo.append(r)
    except Exception as e:
        dataset_completo.append({
            'id': par['id'], 'tipo': par['tipo'], 'dificuldade': par['dificuldade'],
            'question': par['question'], 'answer': f'[Erro: {e}]',
            'contexts': ['Contexto indisponível.'], 'ground_truth': par['ground_truth'],
        })

tot = time.time() - t0
print(f'\n✓ {len(dataset_completo)} pares processados em {tot:.0f}s ({tot/len(dataset_completo):.1f}s/par)')
print(f'  LLM ativo: {LLM_PROVIDER}/{LLM_MODEL}')

## 7. Validação RAGAS (4 campos obrigatórios)

In [ ]:
CAMPOS = ['question', 'answer', 'contexts', 'ground_truth']
problemas = []
for i, p in enumerate(dataset_completo):
    for c in CAMPOS:
        if c not in p or not p[c]:
            problemas.append(f'par {i}: "{c}" ausente/vazio')
        elif c == 'contexts' and (not isinstance(p[c], list) or not p[c]):
            problemas.append(f'par {i}: contexts deve ser lista não-vazia')

print(f'pares validados: {len(dataset_completo)} | problemas: {len(problemas)}')
for x in problemas[:5]:
    print(f'  - {x}')
if not problemas:
    print('✓ todos os pares têm os 4 campos RAGAS preenchidos')

## 8. Exportar JSON + CSV (entrada dos LABs 2–6)

In [ ]:
OUT_JSON = 'dataset_avaliacao_completo.json'
OUT_CSV  = 'dataset_avaliacao_completo.csv'

with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(dataset_completo, f, ensure_ascii=False, indent=2)

df_out = pd.DataFrame(dataset_completo).copy()
df_out['contexts_str'] = df_out['contexts'].apply(lambda lst: ' ||| '.join(lst))
df_out[['id','tipo','dificuldade','question','answer','contexts_str','ground_truth']].to_csv(
    OUT_CSV, index=False, encoding='utf-8')

print(f'✓ {OUT_JSON} ({os.path.getsize(OUT_JSON)/1024:.1f} KB)')
print(f'✓ {OUT_CSV}  ({os.path.getsize(OUT_CSV)/1024:.1f} KB)')

# Salvar config para os próximos labs
with open('lab1_config.json', 'w', encoding='utf-8') as f:
    json.dump({'index_name': INDEX_NAME, 'embed_dim': EMBED_DIM,
                'embed_model': OLLAMA_EMBED_MODEL,
                'llm_provider': LLM_PROVIDER, 'llm_model': LLM_MODEL,
                'k_docs': K_DOCS, 'dataset_path': OUT_JSON}, f, indent=2)
print('✓ lab1_config.json')

## 9. Estatísticas Descritivas

In [ ]:
import statistics as st
ans_len = [len(p['answer']) for p in dataset_completo]
ctx_n   = [len(p['contexts']) for p in dataset_completo]
print('=== Métricas do Dataset Gerado ===')
print(f'  comprimento médio answer: {st.mean(ans_len):.0f} chars (min={min(ans_len)} max={max(ans_len)})')
print(f'  ctxs por par (média)    : {st.mean(ctx_n):.1f}')
print('\n  distribuição por tipo:')
for k, v in pd.DataFrame(dataset_completo)['tipo'].value_counts().items():
    print(f'    {k:25s} {v:>3}')
print('\n  distribuição por dificuldade:')
for k, v in pd.DataFrame(dataset_completo)['dificuldade'].value_counts().items():
    print(f'    {k:10s} {v:>3}')

## Checklist

| # | Item | OK |
|---|------|----|
| 1 | Stack Groq/Ollama/OpenSearch conectada | ☐ |
| 2 | Índice TCU 2026 disponível com embeddings BGE-M3 (1024d) | ☐ |
| 3 | Naive RAG executado em 50 pares | ☐ |
| 4 | 4 campos RAGAS validados | ☐ |
| 5 | `dataset_avaliacao_completo.json` + `.csv` exportados | ☐ |
| 6 | `lab1_config.json` gravado para LABs 2–6 | ☐ |

## Referências

ES, S. et al. **RAGAS**. arXiv:2309.15217, 2023.  
GROQ INC. **Groq API Documentation**. <https://console.groq.com/docs>.  
OLLAMA. **bge-m3 model**. <https://ollama.com/library/bge-m3>.  
OPENSEARCH PROJECT. **Hybrid / Vector Search**. <https://docs.opensearch.org/3.0/vector-search/>.